# A-1 · Competitor impact: who was hit, and did they come back? (Q1)
**Stream A, Person A** · feeds slide 4 · method from `docs/who_answers_what.md`

**Questions**
1. When did sales change? Confirm the working competitor date (`C.COMPETITOR_OPEN` = March 2024).
2. Which age bands, categories and areas were hit hardest?
3. Why did the hardest-hit group fall: fewer customers, fewer visits, or smaller baskets?
4. Did they come back in 2025–2026?

**Method rules**
- **Purchases only** (`row_type == C.PURCHASE`), revenue = quantity × price paid.
- **Year-over-year (YoY)** comparisons only: each month vs. the same month a year earlier, so seasonality (Black Friday, Boxing Week, summer) cancels out.
- **Comparison window:** Mar–Oct 2024 vs. Mar–Oct 2023. Loyalty launched in June 2024, inside that window, so we also check **Mar–May**, the 3 months with the competitor but no loyalty program yet.

In [ ]:
import sys
from pathlib import Path

# make the repo root importable (this notebook lives in analysis/a_competitor/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.metrics import load
from src import config as C

OUT = C.RESULTS_DIR / "a_competitor"
OUT.mkdir(parents=True, exist_ok=True)

AGE_ORDER = ["18-24", "25-34", "35-44", "45-54", "55+"]       # "Unknown" (2% of revenue) left out of charts
AGE_COLORS = {"18-24": "#D1495B", "25-34": "#E09F3E", "35-44": "#2B4C7E",
              "45-54": "#5B8DB8", "55+": "#9AA5B1"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold",
                     "axes.titlelocation": "left", "axes.grid": True, "grid.alpha": 0.25,
                     "legend.frameon": False})

lines = load("lines")
visits = load("visits")
customers = load("customers")

p = lines[lines.row_type == C.PURCHASE].copy()
p["year"], p["mon"] = p.date.dt.year, p.date.dt.month
print(f"{len(p):,} purchase lines, ${p.revenue.sum():,.0f} purchase revenue")

## 1. Year-over-year revenue by age band

In [ ]:
monthly = p.pivot_table(index="month", columns="age_band", values="revenue", aggfunc="sum")
monthly["All customers"] = monthly.sum(axis=1)
yoy = monthly.pct_change(12, fill_method=None).loc["2023-11-01":]   # YoY needs 12 months of history

fig, ax = plt.subplots(figsize=(11, 4.8))
for age in AGE_ORDER:
    hit = age == "18-24"
    ax.plot(yoy.index, yoy[age], color=AGE_COLORS[age], lw=2.6 if hit else 1.3,
            alpha=1 if hit else 0.8, label=age, zorder=3 if hit else 2)
ax.plot(yoy.index, yoy["All customers"], color="black", lw=1.4, ls=":", label="All customers")
ax.axhline(0, color="#333", lw=0.8)
top = ax.get_ylim()[1]
for (d, name), y in zip([(C.COMPETITOR_OPEN, "Competitor opens\n(Mar 2024)"), (C.LOYALTY_LAUNCH, "Loyalty\nlaunch")],
                        [top * 0.97, top * 0.72]):
    ax.axvline(d, color="#555", ls="--", lw=0.9)
    ax.text(d, y, " " + name, fontsize=8.5, va="top")
# From Mar 2025, each month is compared with a post-competitor month (a low base), so YoY jumps partly reflect that
ax.axvspan(pd.Timestamp("2025-03-01"), yoy.index.max(), color="#9AA5B1", alpha=0.08)
ax.text(pd.Timestamp("2025-03-15"), ax.get_ylim()[0] * 0.92,
        "From Mar 2025: compared with post-competitor months (low base)", fontsize=8, color="#555")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set(title="From March 2024, 18–24 revenue ran 40–55% below the year before; other ages dipped then recovered",
       ylabel="Revenue vs. same month last year", xlabel="")
ax.legend(ncol=6, loc="lower center", bbox_to_anchor=(0.5, -0.25), fontsize=9)
plt.tight_layout(); fig.savefig(OUT / "a1_yoy_by_age.png", bbox_inches="tight"); plt.show()

### Confirming the competitor date
The working date is March 2024. If the competitor opened then, the YoY line should be roughly flat or positive up to February and drop sharply from March, starting with the youngest customers. The table shows Nov 2023 – Jun 2024.

In [ ]:
yoy_tbl = yoy.loc["2023-11-01":"2024-06-01", AGE_ORDER + ["All customers"]]
yoy_tbl.index = yoy_tbl.index.strftime("%b %Y")
display(yoy_tbl.style.format("{:+.0%}").background_gradient(cmap="RdYlGn", vmin=-0.5, vmax=0.5))

first_break = yoy.loc[yoy["18-24"] < -0.20].index.min()
print(f"First month 18-24 YoY falls below -20%: {first_break:%B %Y}  (working date: {C.COMPETITOR_OPEN:%B %Y})")
print("Confirmed: the data supports March 2024." if first_break == C.COMPETITOR_OPEN
      else "Differs from the working date. Tell the team at the check-in (config.py stays frozen).")

## 2. Who was hit hardest?

In [ ]:
def window(df, year, m0=3, m1=10):
    return df[(df.year == year) & df.mon.between(m0, m1)]

def yoy_by(key, m0=3, m1=10):
    base = window(p, 2023, m0, m1).groupby(key).revenue.sum()
    hit = window(p, 2024, m0, m1).groupby(key).revenue.sum()
    later = window(p, 2025, m0, m1).groupby(key).revenue.sum()
    return pd.DataFrame({
        "revenue 2023": base,
        "share of 2023": base / base.sum(),
        "2024 vs 2023": hit / base - 1,
        "lost $ (2024)": hit - base,
        "2025 vs 2023": later / base - 1,
    }).sort_values("2024 vs 2023")

fmt = {"revenue 2023": "${:,.0f}", "share of 2023": "{:.0%}", "2024 vs 2023": "{:+.0%}",
       "lost $ (2024)": "${:,.0f}", "2025 vs 2023": "{:+.0%}"}
tables = {k: yoy_by(k) for k in ["age_band", "category", "home_area", "acquisition_channel"]}
for k, t in tables.items():
    print(f"\n=== {k}: Mar–Oct, each year vs. Mar–Oct 2023 ===")
    display(t.style.format(fmt))

In [ ]:
# Robustness: Mar–May only (competitor open, loyalty not launched yet)
clean = pd.DataFrame({"Mar–Oct": tables["age_band"]["2024 vs 2023"],
                      "Mar–May (before loyalty)": yoy_by("age_band", 3, 5)["2024 vs 2023"]})
display(clean.loc[AGE_ORDER].style.format("{:+.0%}"))

In [ ]:
# Is the category drop really an age effect? Category YoY inside each age band (Mar–Oct 2024 vs 2023)
a = window(p, 2023).pivot_table(index="age_band", columns="category", values="revenue", aggfunc="sum")
b = window(p, 2024).pivot_table(index="age_band", columns="category", values="revenue", aggfunc="sum")
age_cat = (b / a - 1).loc[AGE_ORDER]
display(age_cat.style.format("{:+.0%}").background_gradient(cmap="RdYlGn", vmin=-0.5, vmax=0.5, axis=None))

lost = (b - a).loc[AGE_ORDER]
print(f"Leggings lost ${-lost['Leggings'].sum():,.0f}; only "
      f"{lost.loc['18-24', 'Leggings'] / lost['Leggings'].sum():.0%} of that loss came from 18-24 customers")
print(f"Leggings fell in every age band: {age_cat['Leggings'].min():+.0%} to {age_cat['Leggings'].max():+.0%} "
      "-> a category-wide hit on a staple, on top of the 18-24 loss")
print(f"18-24 share of all lost revenue: {lost.loc['18-24'].sum() / lost.sum().sum():.0%} "
      f"(they were {tables['age_band'].loc['18-24', 'share of 2023']:.0%} of 2023 revenue)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, key, title, n_hi in [(axes[0], "category", "By category", 1), (axes[1], "home_area", "By home area", 2)]:
    t = tables[key]["2024 vs 2023"].drop("Unknown", errors="ignore").sort_values()
    colors = ["#D1495B" if i < n_hi else "#9AA5B1" for i in range(len(t))]
    ax.barh(t.index, t.values, color=colors)
    for i, v in enumerate(t.values):
        ax.text(v + 0.004, i, f"{v:.0%}", va="center", ha="left", fontsize=8.5, color="white", fontweight="bold")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    ax.set(title=title, xlabel="Mar–Oct 2024 vs. Mar–Oct 2023")
    ax.invert_yaxis()
fig.suptitle("Leggings fell hardest (−30%), across every age group; Burnaby and Richmond customers pulled back most",
             x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); fig.savefig(OUT / "a1_hardest_hit_category_area.png", bbox_inches="tight"); plt.show()

## 3. Why did 18–24 revenue fall?
Revenue = **active customers × visits per customer × spend per visit**. Breaking the drop down this way shows whether the competitor took whole customers, took some of their trips, or made baskets smaller.

In [ ]:
v = (visits[visits.has_purchase]
     .merge(customers[["customer_id", "age_band"]], on="customer_id", validate="many_to_one"))
v["year"], v["mon"] = v.date.dt.year, v.date.dt.month

def drivers(year):
    w = window(v, year)
    g = w.groupby("age_band").agg(customers=("customer_id", "nunique"),
                                  visits=("transaction_id", "size"), revenue=("revenue", "sum"))
    g["visits per customer"] = g.visits / g.customers
    g["spend per visit"] = g.revenue / g.visits
    return g.loc[AGE_ORDER, ["customers", "visits per customer", "spend per visit", "revenue"]]

d23, d24, d25 = drivers(2023), drivers(2024), drivers(2025)
change24 = d24 / d23 - 1
change25 = d25 / d23 - 1
display(pd.concat({"Mar–Oct 2023": d23, "2024 vs 2023": change24, "2025 vs 2023": change25}, axis=1)
        .style.format("{:,.2f}").format("{:+.0%}", subset=["2024 vs 2023", "2025 vs 2023"]))

In [ ]:
metrics = ["customers", "visits per customer", "spend per visit"]
fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(len(metrics)); w = 0.36
for i, (lab, ch, col) in enumerate([("2024 vs 2023", change24, "#D1495B"), ("2025 vs 2023", change25, "#2B4C7E")]):
    vals = ch.loc["18-24", metrics].values
    ax.bar(x + (i - 0.5) * w, vals, w, color=col, label=lab)
    for xi, val in zip(x + (i - 0.5) * w, vals):
        label = "0%" if abs(val) < 0.005 else f"{val:+.0%}"
        ax.text(xi, val + (0.01 if val >= 0 else -0.01), label, ha="center",
                va="bottom" if val >= 0 else "top", fontsize=9)
ax.axhline(0, color="#333", lw=0.8)
ax.set_xticks(x, ["Active customers", "Visits per customer", "Spend per visit"])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set(title="18–24s: the competitor took customers and trips, not basket size; lost customers never came back",
       ylabel="vs. Mar–Oct 2023")
ax.legend()
plt.tight_layout(); fig.savefig(OUT / "a1_18_24_drivers.png", bbox_inches="tight"); plt.show()

## 4. Did they come back?

In [ ]:
rev = pd.DataFrame({y: window(p, y).groupby("age_band").revenue.sum() for y in [2023, 2024, 2025, 2026]}).loc[AGE_ORDER]
idx = rev.div(rev[2023], axis=0) * 100      # index: Mar–Oct 2023 = 100

fig, ax = plt.subplots(figsize=(9, 4))
for age in AGE_ORDER:
    hit = age == "18-24"
    ax.plot(idx.columns.astype(str), idx.loc[age], marker="o", color=AGE_COLORS[age],
            lw=2.6 if hit else 1.4, label=age)
    ax.text(3.08, idx.loc[age, 2026], f"{age}  {idx.loc[age, 2026]:.0f}", va="center", fontsize=8.5,
            color=AGE_COLORS[age], fontweight="bold" if hit else "normal")
ax.axhline(100, color="#333", lw=0.8)
ax.set(title="Every age band recovered except 18–24, still ~45% below 2023",
       ylabel="Mar–Oct revenue (2023 = 100)", xlim=(-0.2, 3.6))
plt.tight_layout(); fig.savefig(OUT / "a1_recovery_index.png", bbox_inches="tight"); plt.show()
idx.round(0)

## 5. Headline numbers to hand over

In [ ]:
age = tables["age_band"]; cat = tables["category"]; area = tables["home_area"]
total23 = window(p, 2023).revenue.sum(); total24 = window(p, 2024).revenue.sum()
total25 = window(p, 2025).revenue.sum()
summary = {
    "Competitor date supported by data": f"{first_break:%B %Y}",
    "Store revenue Mar–Oct 2024 vs 2023": f"{total24 / total23 - 1:+.0%} (${total24 - total23:,.0f})",
    "Store revenue Mar–Oct 2025 vs 2023": f"{total25 / total23 - 1:+.0%}",
    "18-24 revenue 2024 vs 2023": f"{age.loc['18-24', '2024 vs 2023']:+.0%}",
    "18-24 revenue 2025 vs 2023": f"{age.loc['18-24', '2025 vs 2023']:+.0%}",
    "18-24 share of all lost revenue (2024)": f"{lost.loc['18-24'].sum() / lost.sum().sum():.0%}",
    "18-24 active customers 2024 / 2025 vs 2023": f"{change24.loc['18-24', 'customers']:+.0%} / {change25.loc['18-24', 'customers']:+.0%}",
    "18-24 spend per visit 2024 vs 2023": f"{change24.loc['18-24', 'spend per visit']:+.0%}",
    "Hardest-hit category": f"{cat.index[0]} {cat.iloc[0]['2024 vs 2023']:+.0%}",
    "Hardest-hit areas": ", ".join(f"{i} {r['2024 vs 2023']:+.0%}" for i, r in area.drop('Unknown').head(2).iterrows()),
    "Outerwear 2025 vs 2023 (for Stream C)": f"{cat.loc['Outerwear', '2025 vs 2023']:+.0%} overall",
}
# For Stream C: is the 2025 outerwear dip an age effect rather than repair cannibalization?
c25 = window(p, 2025).pivot_table(index="age_band", columns="category", values="revenue", aggfunc="sum")
ow = (c25 / a - 1).loc[AGE_ORDER, "Outerwear"]
summary["Outerwear 2025 vs 2023 by age (for Stream C)"] = ", ".join(f"{k} {v:+.0%}" for k, v in ow.items())
pd.Series(summary, name="value").to_frame()